# 03. Feature Engineering

## Overview
Phase 3 builds calendar, lag, and rolling features while **strictly preventing data leakage**.

### Leakage Prevention Rule:
All rolling features use `.shift(1)` before window aggregation.
Example:
`df["rolling_mean_6"] = df["Appliances"].shift(1).rolling(6).mean()`

### Features Created:
- **Calendar**: `year`, `month`, `day`, `hour`, `day_of_week`, `is_weekend`
- **Lags**: `lag_6` (1 hr), `lag_144` (1 day), `lag_1008` (1 week)
- **Rolling**: `rolling_mean_6` (1 hr rolling avg), `rolling_mean_144` (1 day rolling avg)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "Data" / "energy_consumption_cleaned.csv"
FEATURES_OUTPUT = PROJECT_ROOT / "Data" / "energy_consumption_features.csv"
FEATURE_LIST_PATH = PROJECT_ROOT / "outputs" / "feature_list.txt"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Cleaned dataset loaded. Initial shape:", df.shape)


Cleaned dataset loaded. Initial shape: (19735, 29)


In [2]:
# 1. Calendar Features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# 2. Lag Features (10-min interval steps: 6=1h, 144=1d, 1008=7d)
df["lag_6"] = df["Appliances"].shift(6)
df["lag_144"] = df["Appliances"].shift(144)
df["lag_1008"] = df["Appliances"].shift(1008)

# 3. Rolling Features (shift(1) prevents data leakage)
df["rolling_mean_6"] = df["Appliances"].shift(1).rolling(6).mean()
df["rolling_mean_144"] = df["Appliances"].shift(1).rolling(144).mean()

print("Engineered calendar, lag, and rolling features.")


Engineered calendar, lag, and rolling features.


In [3]:
# Drop NaN rows resulting from shifts
df_model = df.dropna().reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Feature dataset shape after dropna: {df_model.shape}")
df_model[["date", "Appliances", "lag_6", "lag_144", "lag_1008", "rolling_mean_6", "rolling_mean_144"]].head(10)


Original shape: (19735, 40)
Feature dataset shape after dropna: (18727, 40)


,date,Appliances,lag_6,lag_144,lag_1008,rolling_mean_6,rolling_mean_144
0,2016-01-18 17:00:00,40,50.0,90.0,60.0,46.666667,79.305556
1,2016-01-18 17:10:00,40,50.0,130.0,60.0,45.000000,78.958333
2,2016-01-18 17:20:00,40,50.0,380.0,50.0,43.333333,78.333333
3,2016-01-18 17:30:00,40,40.0,800.0,50.0,41.666667,75.972222
4,2016-01-18 17:40:00,20,40.0,790.0,60.0,41.666667,70.694444
5,2016-01-18 17:50:00,30,50.0,540.0,50.0,38.333333,65.347222
6,2016-01-18 18:00:00,30,40.0,120.0,60.0,35.000000,61.805556
7,2016-01-18 18:10:00,70,40.0,110.0,60.0,33.333333,61.180556
8,2016-01-18 18:20:00,230,40.0,160.0,60.0,38.333333,60.902778
9,2016-01-18 18:30:00,660,40.0,140.0,70.0,70.000000,61.388889


In [4]:
feature_cols = [
    "lights", "T1", "RH_1", "T2", "RH_2", "T3", "RH_3", "T4", "RH_4", "T5", "RH_5",
    "T6", "RH_6", "T7", "RH_7", "T8", "RH_8", "T9", "RH_9", "T_out", "Press_mm_hg",
    "RH_out", "Windspeed", "Visibility", "Tdewpoint", "year", "month", "day", "hour",
    "day_of_week", "is_weekend", "lag_6", "lag_144", "lag_1008", "rolling_mean_6", "rolling_mean_144"
]

df_model.to_csv(FEATURES_OUTPUT, index=False)
print(f"Saved engineered feature dataset to: {FEATURES_OUTPUT}")

with open(FEATURE_LIST_PATH, "w") as f:
    for feat in feature_cols:
        f.write(f"{feat}\n")
print(f"Saved feature list ({len(feature_cols)} columns) to: {FEATURE_LIST_PATH}")


Saved engineered feature dataset to: C:\Users\admin\Downloads\OneDrive\python\Appliances Energy Prediction\Data\energy_consumption_features.csv
Saved feature list (36 columns) to: C:\Users\admin\Downloads\OneDrive\python\Appliances Energy Prediction\outputs\feature_list.txt


### Feature Engineering Summary:
- Total engineered features: 36 input variables.
- Leakage audited: No current or future actual target values used in rolling windows.
- Saved output: `Data/energy_consumption_features.csv` (18,727 rows).
